# SAM-AI Sovereign Agent: Real-Model Smoke Test

**Objective:** Verify that the sovereign SAM-AI agent (`sam_ai.agent`) loads authentic model weights, connects to official ARC-3 environment files, and executes a strictly budgeted 5-step interactive loop.

> **Hard Rules:**
> 1. Strictly unsubmitted: Does not create `submission.parquet` or call competition submission APIs.
> 2. No mock fallback: If model weights, GPU accelerator, or environment data are absent, halts cleanly with `[BLOCKED]`.
> 3. Small compute footprint: Bounded to 5 environment steps total (< 2 minutes GPU runtime).


In [ ]:
# ==============================================================================
# 1. Environment & Prerequisite Audit
# ==============================================================================
import os
import sys
import time
import json
from pathlib import Path

print('=' * 70)
print('  SAM-AI REAL-MODEL SMOKE TEST: PRECONDITION AUDIT')
print('=' * 70)

missing = []

# 1. Check GPU
try:
    import torch
    if not torch.cuda.is_available():
        missing.append('CUDA is not available. GPU accelerator must be enabled in Settings.')
    else:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f'[OK] Detected GPU: {gpu_name}')
        print(f'[OK] Total VRAM:   {vram_gb:.2f} GB')
except ImportError:
    missing.append('PyTorch is not installed.')

# 2. Check Base Model Path
BASE_MODEL_CANDIDATES = [
    '/kaggle/input/deepseek-r1-distill-qwen-14b-bnb-4bit',
    '/kaggle/input/models/unsloth/deepseek-r1-distill-qwen-14b-bnb-4bit',
    '/kaggle/input/qwen2.5-coder-14b-instruct',
    'unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit',
]
base_model_path = None
for c in BASE_MODEL_CANDIDATES:
    if Path(c).exists():
        base_model_path = c
        break

if not base_model_path:
    missing.append('Base model weights not found in /kaggle/input/ mounts.')
else:
    print(f'[OK] Base Model Path: {base_model_path}')

# 3. Check Adapter Weights (Checkpoint-25 Verified)
ADAPTER_CANDIDATES = [
    '/kaggle/input/sam-ai-v4-adapter-checkpoint-25',
    'kaggle_flywheel_check/sam_ai_v4_trained/checkpoint-25',
]
adapter_path = None
for a in ADAPTER_CANDIDATES:
    if Path(a).exists():
        weights = list(Path(a).glob('*.safetensors')) + list(Path(a).glob('*.bin'))
        if weights and sum(w.stat().st_size for w in weights) > 0:
            adapter_path = a
            break

print(f'[INFO] Adapter Path: {adapter_path or "None (Evaluating Zero-Shot Base Model)"}')

# 4. Check ARC-3 Public Environment Data
ENV_DATA_DIR = '/kaggle/input/arc-prize-2026-arc-agi-3/environment_files'
if not Path(ENV_DATA_DIR).exists() or not list(Path(ENV_DATA_DIR).rglob('metadata.json')):
    missing.append(f'ARC-3 environment data not found at {ENV_DATA_DIR}. Attach arc-prize-2026-arc-agi-3.')
else:
    found_envs = len(list(Path(ENV_DATA_DIR).rglob('metadata.json')))
    print(f'[OK] Found {found_envs} official ARC-3 environment files.')

if missing:
    print('\n[BLOCKED] Cannot execute real-model smoke test.')
    for m in missing:
        print(f'  - {m}')
    raise RuntimeError('Smoke test halted due to missing prerequisites.')

print('\n[READY] All hardware, model, and environment prerequisites verified.')


In [ ]:
# ==============================================================================
# 2. Model Loading & Verification
# ==============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print(f'[*] Loading tokenizer from {base_model_path}...')
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'[*] Loading model weights into GPU memory...')
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True,
)

if adapter_path:
    print(f'[*] Attaching PEFT adapter: {adapter_path}...')
    model = PeftModel.from_pretrained(base_model, adapter_path)
else:
    model = base_model

model.eval()
device = next(model.parameters()).device
print(f'[OK] Model successfully loaded on device: {device}')


In [ ]:
# ==============================================================================
# 3. ModelRunner Wrapper & SAM-AI Agent Setup
# ==============================================================================
from sam_ai.substrate.interfaces import ModelRunner, ModelOutput
from sam_ai.agent.arc3_agent import SAMARC3Agent

class KaggleInProcessRunner(ModelRunner):
    def __init__(self, model, tokenizer, name):
        self.model = model
        self.tokenizer = tokenizer
        self._name = name

    @property
    def model_name(self) -> str:
        return self._name

    def generate(self, prompt, system_prompt=None, temperature=0.0, max_tokens=1024, stop=None):
        t0 = time.perf_counter()
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})
        
        if hasattr(self.tokenizer, 'apply_chat_template') and self.tokenizer.chat_template:
            fmt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            sys_p = f'System: {system_prompt}\n\n' if system_prompt else ''
            fmt = f'{sys_p}User: {prompt}\n\nAssistant:'
        
        inputs = self.tokenizer(fmt, return_tensors='pt').to(next(self.model.parameters()).device)
        p_tokens = inputs['input_ids'].shape[1]
        
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=(temperature > 0.0),
                temperature=temperature if temperature > 0.0 else None,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        
        comp_ids = out[0][p_tokens:]
        text = self.tokenizer.decode(comp_ids, skip_special_tokens=True)
        latency = (time.perf_counter() - t0) * 1000.0
        return ModelOutput(
            text=text,
            prompt_tokens=p_tokens,
            completion_tokens=len(comp_ids),
            latency_ms=latency,
        )

runner = KaggleInProcessRunner(model, tokenizer, name=f'{base_model_path}+{adapter_path}')

# Initialize SAM-AI Agent (Arm B: Verification Guard Enabled)
agent = SAMARC3Agent(
    model_runner=runner,
    enable_verification_guard=True,
    enable_failure_memory=False,
    is_real_model=True,
)
print('[OK] SAM-AI Agent initialized.')


In [ ]:
# ==============================================================================
# 4. Interactive Execution (Max 5 Steps) & Telemetry
# ==============================================================================
from arc_agi import ArcAgi

client = ArcAgi(data_dir=ENV_DATA_DIR)
all_games = [p.parent.name for p in Path(ENV_DATA_DIR).rglob('metadata.json')]
chosen_game = all_games[0]
print(f'[*] Launching smoke test on game: {chosen_game}')

env = client.get_environment(chosen_game)
summary = agent.run_session(env, game_id=chosen_game, max_steps=5)

# Output telemetry
out_dir = Path('evidence/real_model_smoke_test')
out_dir.mkdir(parents=True, exist_ok=True)

report = {
    'game_id': chosen_game,
    'steps_taken': summary.total_steps,
    'levels_completed': summary.levels_completed,
    'final_state': summary.final_state,
    'verification_failures': summary.verification_failures,
    'action_history': summary.action_history,
    'is_real_model': True,
}

with open(out_dir / 'smoke_summary.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print('=' * 70)
print(f'SMOKE TEST COMPLETE: Steps={summary.total_steps}, State={summary.final_state}')
print(f'Summary saved to: {out_dir / "smoke_summary.json"}')
print('=' * 70)
